# Onboarding RAG — QLoRA Fine-tune (Unsloth + Colab)

Fine-tune **Llama 3.1 8B Instruct** on your onboarding Q&A dataset using **4-bit QLoRA** via [Unsloth](https://github.com/unslothai/unsloth).

## Before you run
1. **Runtime → Change runtime type → T4 GPU** (free tier works; A100 is faster)
2. **Runtime → Restart runtime** before first install (clean Colab env)
3. Upload `train.jsonl` and `val.jsonl` from `Fine-tuning/jsonl/` **or** mount Google Drive (see next cell)
4. Expected dataset: **924 train / 171 val** examples
5. **Optional:** set `USE_WANDB = True` in the config cell to log to [Weights & Biases](https://wandb.ai)

## What this does
- Loads base model in **4-bit** (QLoRA — no separate quantize step)
- Attaches **LoRA** adapters
- Trains only on **assistant** tokens
- **Saves checkpoints every 100 steps** to Drive (`lora_output/checkpoints/`)
- **Auto-resumes** from the latest checkpoint if interrupted
- Logs **train/eval loss** to W&B (if enabled)
- Saves final adapter to Drive for RAG inference

In [ ]:
# Install Unsloth (Colab) — run once, auto-restarts, then run this cell again
from pathlib import Path

MARKER = Path("/content/.onboarding_unsloth_ready")

if not MARKER.exists():
    # Do NOT import torch/trl/unsloth before pip — numpy must not load until after restart
    pip = lambda *args: get_ipython().run_line_magic("pip", " ".join(args))

    pip("install", "-q", "--upgrade", "pip")
    # Colab ships torchvision built for a different torch → torchvision::nms crash
    pip("uninstall", "-y", "torchvision", "torchaudio")
    pip("install", "-q", "-U", "trl>=0.18.2,<=0.24.0,!=0.19.0", "datasets>=3.4.1,<4.4.0")
    pip(
        "install",
        "-q",
        "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    )
    pip("install", "-q", "wandb")
    # Reinstall torchvision to match the torch version unsloth just installed
    pip("install", "-q", "torchvision")

    MARKER.write_text("ok")
    import os
    print("Packages installed. Restarting runtime (required — numpy/torch cannot reload mid-session)...")
    os.kill(os.getpid(), 9)

import torch
import torchvision

major, minor = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
print(f"CUDA capability: {major}.{minor}")
print(f"torch={torch.__version__}  torchvision={torchvision.__version__}")

import trl, transformers, datasets
print(f"trl={trl.__version__}  transformers={transformers.__version__}  datasets={datasets.__version__}")
print("Environment ready — continue to the next cell.")


In [ ]:
# ── Paths: pick ONE option ─────────────────────────────────────────────────
from pathlib import Path

# OPTION A: Google Drive (recommended)
USE_DRIVE = True
DRIVE_DATA_DIR = "/content/drive/MyDrive/onboarding_rag/jsonl"   # train.jsonl + val.jsonl here
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/onboarding_rag/lora_output"

# OPTION B: Upload directly to /content/
LOCAL_DATA_DIR = "/content/jsonl"
LOCAL_OUTPUT_DIR = "/content/lora_output"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(DRIVE_DATA_DIR)
    OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
else:
    DATA_DIR = Path(LOCAL_DATA_DIR)
    OUTPUT_DIR = Path(LOCAL_OUTPUT_DIR)

TRAIN_PATH = DATA_DIR / "train.jsonl"
VAL_PATH = DATA_DIR / "val.jsonl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_PATH.exists(), f"Missing {TRAIN_PATH} — upload train.jsonl first"
assert VAL_PATH.exists(), f"Missing {VAL_PATH} — upload val.jsonl first"
print(f"Train: {TRAIN_PATH}")
print(f"Val:   {VAL_PATH}")
print(f"Out:   {OUTPUT_DIR}")

In [ ]:
# ── Training config ────────────────────────────────────────────────────────
MODEL_NAME = "unsloth/Llama-3.1-8B-Instruct"  # 4-bit QLoRA via Unsloth
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH = 2          # use 1 if OOM on T4
GRAD_ACCUM_STEPS = 4          # effective batch = 8
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10
EVAL_STEPS = 50
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 3          # keep last N checkpoints on Drive
SEED = 3407

# ── Checkpoint resume ──────────────────────────────────────────────────────
# True  → auto-resume from latest checkpoint in lora_output/checkpoints/
# False → start fresh (deletes nothing; just ignores existing checkpoints)
# "path/to/checkpoint-200" → resume from a specific checkpoint folder
RESUME_FROM_CHECKPOINT = True

# ── Weights & Biases ─────────────────────────────────────────────────────
USE_WANDB = True
WANDB_PROJECT = "onboarding-rag-qlora"
WANDB_RUN_NAME = "llama-3.1-8b-lora-v1"   # change per run

In [ ]:
# ── W&B login (Colab) ───────────────────────────────────────────────────────
# API key: https://wandb.ai/authorize
# Colab Secrets (recommended): add WANDB_API_KEY, then uncomment:
# from google.colab import userdata
# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

import os
import wandb

if USE_WANDB:
    os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

    if not os.environ.get("WANDB_API_KEY"):
        from getpass import getpass
        os.environ["WANDB_API_KEY"] = getpass("Enter your W&B API key: ")

    wandb.login()
    print(f"W&B → project: {WANDB_PROJECT} | run: {WANDB_RUN_NAME}")
else:
    print("W&B disabled (USE_WANDB = False)")


In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────
import json
from datasets import Dataset

def load_messages_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                ex = json.loads(line)
                rows.append({"messages": ex["messages"]})
    return rows

train_rows = load_messages_jsonl(TRAIN_PATH)
val_rows = load_messages_jsonl(VAL_PATH)

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

print(f"Train examples: {len(train_ds)}")
print(f"Val examples:   {len(val_ds)}")
print("Sample roles:", [m["role"] for m in train_ds[0]["messages"]])

In [ ]:
# ── Load 4-bit model + LoRA ────────────────────────────────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,              # auto (float16 on T4)
    load_in_4bit=True,       # QLoRA — quantize on load
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# ── Chat template (Llama 3.1) ────────────────────────────────────────────────
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in examples["messages"]
    ]
    return {"text": texts}

train_ds = train_ds.map(formatting_prompts_func, batched=True)
val_ds = val_ds.map(formatting_prompts_func, batched=True)

print(train_ds[0]["text"][:600], "...\n")

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────
import torch
import wandb
from unsloth.chat_templates import train_on_responses_only

# TRL API: SFTConfig (>=0.11) or TrainingArguments (older)
try:
    from trl import SFTTrainer, SFTConfig
    _USE_SFT_CONFIG = True
except ImportError:
    from trl import SFTTrainer
    from transformers import TrainingArguments
    _USE_SFT_CONFIG = False
    print("Note: using TrainingArguments (upgrade trl>=0.11 for SFTConfig)")

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def get_latest_checkpoint(checkpoint_dir: Path) -> Path | None:
    """Return newest checkpoint-* folder that has trainer_state.json."""
    candidates = sorted(
        checkpoint_dir.glob("checkpoint-*"),
        key=lambda p: int(p.name.rsplit("-", 1)[-1]),
    )
    for ckpt in reversed(candidates):
        if (ckpt / "trainer_state.json").exists():
            return ckpt
    return None


def resolve_resume_checkpoint() -> str | None:
    if RESUME_FROM_CHECKPOINT is False:
        return None
    if RESUME_FROM_CHECKPOINT is True:
        latest = get_latest_checkpoint(CHECKPOINT_DIR)
        if latest:
            print(f"Auto-resuming from: {latest}")
        else:
            print("No checkpoint found — starting fresh")
        return str(latest) if latest else None
    ckpt = Path(RESUME_FROM_CHECKPOINT)
    if not ckpt.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt}")
    print(f"Resuming from: {ckpt}")
    return str(ckpt)

_common = dict(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_ratio=WARMUP_RATIO,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="adamw_8bit",
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="linear",
    seed=SEED,
    report_to="wandb" if USE_WANDB else "none",
    run_name=WANDB_RUN_NAME if USE_WANDB else None,
)

if _USE_SFT_CONFIG:
    sft_args = SFTConfig(
        **_common,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
    )
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=sft_args,
    )
else:
    sft_args = TrainingArguments(**_common)
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        args=sft_args,
    )

if USE_WANDB:
    wandb.config.update({
        "model": MODEL_NAME,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "max_seq_length": MAX_SEQ_LENGTH,
        "train_size": len(train_ds),
        "val_size": len(val_ds),
    })

def _llama31_header(role: str) -> str:
    return "<|" + "start_header_id" + "|>" + role + "<|" + "end_header_id" + "|>" + "\n\n"

trainer = train_on_responses_only(
    trainer,
    instruction_part=_llama31_header("user"),
    response_part=_llama31_header("assistant"),
)

gpu_stats = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_stats.name}, {gpu_stats.total_memory / 1e9:.1f} GB")

# Fix: Colab TRL install sometimes omits templates/lm_model_card.md (crashes at end of train)
import types
import trl

def _ensure_trl_model_card_template() -> None:
    tpl_file = Path(trl.__file__).parent / "templates" / "lm_model_card.md"
    if tpl_file.exists():
        return
    tpl_file.parent.mkdir(parents=True, exist_ok=True)
    tpl_file.write_text(
        "---\n{{ card_data }}\n---\n\n# {{ model_name }}\n\n"
        "Fine-tuned with {{ trainer_name }}.\n",
        encoding="utf-8",
    )
    print(f"Wrote missing TRL template: {tpl_file}")

_ensure_trl_model_card_template()

# Saving LoRA locally — skip HF model card if template is still missing
_tpl = Path(trl.__file__).parent / "templates" / "lm_model_card.md"
if not _tpl.exists():
    trainer.create_model_card = types.MethodType(lambda self, **kwargs: None, trainer)
    print("Skipping model card creation (TRL template missing)")

existing = sorted(CHECKPOINT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.rsplit("-", 1)[-1]))
if existing:
    print("Checkpoints on Drive:", [p.name for p in existing])

resume_ckpt = resolve_resume_checkpoint()
trainer_stats = trainer.train(resume_from_checkpoint=resume_ckpt)
print(trainer_stats)

if USE_WANDB and wandb.run is not None:
    run_url = wandb.run.get_url()
    wandb.finish()
    print(f"View run: {run_url}")


In [ ]:
# ── Save LoRA adapter ────────────────────────────────────────────────────────
LORA_SAVE = OUTPUT_DIR / "onboarding_lora"
model.save_pretrained(str(LORA_SAVE))
tokenizer.save_pretrained(str(LORA_SAVE))
print(f"Saved LoRA adapter to: {LORA_SAVE}")

In [ ]:
# ── Quick inference test ─────────────────────────────────────────────────────
FastLanguageModel.for_inference(model)

# Use raw val file so roles are preserved after formatting map
val_raw = load_messages_jsonl(VAL_PATH)[0]["messages"]
messages = [
    {"role": "system", "content": val_raw[0]["content"]},
    {"role": "user", "content": val_raw[1]["content"]},
]
expected = val_raw[2]["content"]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.2,
    use_cache=True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("\n--- EXPECTED ---")
print(expected[:500])

In [ ]:
# ── OPTIONAL: merge LoRA into 16-bit model for easier deployment ─────────────
# Warning: ~16GB disk; skip if you only need the adapter
MERGE_AND_SAVE = False

if MERGE_AND_SAVE:
    merged_path = OUTPUT_DIR / "onboarding_lora_merged_16bit"
    model.save_pretrained_merged(str(merged_path), tokenizer, save_method="merged_16bit")
    print(f"Merged model saved to: {merged_path}")

## Pip dependency warnings (Colab)

After install you may still see warnings about `cudf`, `pandas`, `numba`, `google-colab`, etc. Those are **Colab pre-installed packages** — safe to ignore for Unsloth training.

**Do fix** warnings about `unsloth-zoo` vs `trl` / `transformers` / `datasets` / `torch`. They mean versions are incompatible and training may crash. The install cell pins:
- `trl` 0.18.2–0.24.0 (has `SFTConfig`; **not** trl 1.x)
- `datasets` 3.4.1–4.3.x

Never run `pip install --force-reinstall "trl>=0.11.0"` — that upgrades to trl 1.8+ and breaks Unsloth.

## Checkpointing & resume

Checkpoints are saved to `lora_output/checkpoints/checkpoint-{step}/` on Drive every **100 steps**.

| `RESUME_FROM_CHECKPOINT` | Behavior |
|---|---|
| `True` (default) | Auto-resume from latest checkpoint if one exists |
| `False` | Start a fresh run (ignores existing checkpoints) |
| `"/content/drive/.../checkpoint-200"` | Resume from a specific folder |

After a Colab disconnect: **Runtime → Restart** → re-run all cells. With `RESUME_FROM_CHECKPOINT = True`, training continues from the last saved step.

To inspect checkpoints without training:
```python
from pathlib import Path
sorted((OUTPUT_DIR / "checkpoints").glob("checkpoint-*"))
```

## OOM troubleshooting (T4 16GB)

If you get CUDA out-of-memory:
1. Set `PER_DEVICE_BATCH = 1`
2. Set `MAX_SEQ_LENGTH = 1536`
3. Set `GRAD_ACCUM_STEPS = 8` (keeps effective batch size)

## After training

Download `onboarding_lora/` from Drive. At RAG inference time:
1. Load `unsloth/Llama-3.1-8B-Instruct` in 4-bit
2. Load your LoRA adapter
3. Pass retrieved context using the same `Context (type):` + metadata format from `build_jsonl.py`